In [0]:
from pyspark.sql.functions import col, trim, upper, when, sum as spark_sum
from pyspark.sql.functions import broadcast

#Load bronze tables
students_df       = spark.table("bronze.alunos")
municipio_df      = spark.table("bronze.municipio")
uf_df             = spark.table("bronze.uf")
goal_brazil_df    = spark.table("bronze.meta_alfabetizacao_brasil")
goal_uf_df        = spark.table("bronze.meta_alfabetizacao_uf")
goal_municipio_df = spark.table("bronze.meta_alfabetizacao_municipio")
directory_df      = spark.table("bronze.municipio_directory")

#Cleaning and trimming function
def clean_dataframe(df, string_cols):
    df = df.dropDuplicates()
    for c in string_cols:
        if c in df.columns:
            df = df.withColumn(c, trim(col(c)))
    return df

#Replace numeric rede codes with their text meaning
def decode_rede(df, table_name, dicionario_df):
    mapping_df = (dicionario_df
        .filter((col("id_tabela") == table_name) & (col("nome_coluna") == "rede"))
        .select(col("chave").alias("rede"), col("valor").alias("rede_nome")))

    df = (df.join(broadcast(mapping_df), on="rede", how="left")
            .drop("rede")
            .withColumnRenamed("rede_nome", "rede"))
    return df

#Apply cleaning functions
students_df  = clean_dataframe(students_df, ["id_municipio", "id_escola", "id_aluno",
                                              "caderno", "serie", "rede",
                                              "presenca", "preenchimento_caderno", "alfabetizado"])
municipio_df = clean_dataframe(municipio_df, ["id_municipio", "serie", "rede"])
uf_df        = clean_dataframe(uf_df, ["sigla_uf", "serie", "rede"])
goal_brazil_df    = clean_dataframe(goal_brazil_df, ["rede"])
goal_uf_df        = clean_dataframe(goal_uf_df, ["sigla_uf", "rede"])
goal_municipio_df = clean_dataframe(goal_municipio_df, ["id_municipio", "rede"])
directory_df      = clean_dataframe(directory_df, ["id_municipio", "sigla_uf", "nome"])

dicionario_df = spark.table("bronze.dicionario")
municipio_df = decode_rede(municipio_df, "municipio", dicionario_df)
uf_df        = decode_rede(uf_df, "uf", dicionario_df)
students_df  = decode_rede(students_df, "alunos", dicionario_df)

#Standardize sigla_uf across multiple sources
uf_df        = uf_df.withColumn("sigla_uf", upper(col("sigla_uf")))
goal_uf_df   = goal_uf_df.withColumn("sigla_uf", upper(col("sigla_uf")))
directory_df = directory_df.withColumn("sigla_uf", upper(col("sigla_uf")))

#Rename columns to avoid column name duplicity
goal_municipio_df = goal_municipio_df.withColumnRenamed("taxa_alfabetizacao", "taxa_alfabetizacao_meta_base")
goal_uf_df         = goal_uf_df.withColumnRenamed("taxa_alfabetizacao", "taxa_alfabetizacao_meta_base")


#Select only the directory columns relevant for enrichment
directory_slim_df = directory_df.select(
    col("id_municipio"),
    col("nome").alias("municipio_nome"),
    col("nome_uf"),
    col("nome_regiao"),
    col("amazonia_legal"),
    col("centroide")
)

#Key validation
orphan_municipalities = (municipio_df.select("id_municipio").distinct()
    .join(directory_slim_df.select("id_municipio"), on="id_municipio", how="left_anti"))
orphan_count = orphan_municipalities.count()
print(f"Municipalities in bronze.municipio without a directory match: {orphan_count}")

#Aggregate data by students
students_weighted_df = students_df.withColumn(
    "alfabetizado_flag",
    when(col("alfabetizado") == "1", 1).otherwise(0)
)

students_aggregated_df = (students_weighted_df
    .groupBy("ano", "id_municipio", "rede")
    .agg(
        (spark_sum(col("alfabetizado_flag") * col("peso_aluno")) / spark_sum("peso_aluno") * 100)
            .alias("taxa_alfabetizacao_recalculada")
    ))

#Build silver tables
silver_municipio_df = (municipio_df
    .join(goal_municipio_df, on=["ano", "id_municipio", "rede"], how="left")
    .join(directory_slim_df, on="id_municipio", how="left")
    .join(students_aggregated_df, on=["ano", "id_municipio", "rede"], how="left"))

silver_municipio_df = silver_municipio_df.withColumn(
    "possui_meta_municipal",
    when(col("meta_alfabetizacao_2024").isNotNull(), True).otherwise(False)
)

silver_uf_df = uf_df.join(goal_uf_df, on=["ano", "sigla_uf", "rede"], how="left")

silver_brazil_df = goal_brazil_df

silver_students_df = students_df

#Write to silver
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

silver_municipio_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.resultado_municipio")

silver_uf_df.write.format("delta").mode("overwrite") \
    .saveAsTable("silver.resultado_uf")

silver_brazil_df.write.format("delta").mode("overwrite") \
    .saveAsTable("silver.resultado_brasil")

silver_students_df.write.format("delta").mode("overwrite") \
    .partitionBy("ano") \
    .saveAsTable("silver.alunos_tratado")

print("Silver layer written: resultado_municipio, resultado_uf, resultado_brasil, alunos_tratado")